 ***EDA***         **and** ***Model Selection***

In [72]:
import numpy as np
import pandas as pd
from scipy.stats import f_oneway,ttest_ind
from scipy.stats import pearsonr
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,TargetEncoder


from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.svm import SVR

from sklearn.model_selection import GridSearchCV,RandomizedSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score,accuracy_score

**Data Loading**

In [43]:
df=pd.read_csv("../Dataset/CarPrice.csv")
df.tail()

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
200,201,-1,volvo 145e (sw),gas,std,four,sedan,rwd,front,109.1,...,141,mpfi,3.78,3.15,9.5,114,5400,23,28,16845.0
201,202,-1,volvo 144ea,gas,turbo,four,sedan,rwd,front,109.1,...,141,mpfi,3.78,3.15,8.7,160,5300,19,25,19045.0
202,203,-1,volvo 244dl,gas,std,four,sedan,rwd,front,109.1,...,173,mpfi,3.58,2.87,8.8,134,5500,18,23,21485.0
203,204,-1,volvo 246,diesel,turbo,four,sedan,rwd,front,109.1,...,145,idi,3.01,3.40,23.0,106,4800,26,27,22470.0
204,205,-1,volvo 264gl,gas,turbo,four,sedan,rwd,front,109.1,...,141,mpfi,3.78,3.15,9.5,114,5400,19,25,22625.0


In [44]:
df.shape

(205, 26)

In [45]:
y=df["price"]
x=df.drop("price",axis=1)

In [46]:
x.shape

(205, 25)

In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 26 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   car_ID            205 non-null    int64  
 1   symboling         205 non-null    int64  
 2   CarName           205 non-null    object 
 3   fueltype          205 non-null    object 
 4   aspiration        205 non-null    object 
 5   doornumber        205 non-null    object 
 6   carbody           205 non-null    object 
 7   drivewheel        205 non-null    object 
 8   enginelocation    205 non-null    object 
 9   wheelbase         205 non-null    float64
 10  carlength         205 non-null    float64
 11  carwidth          205 non-null    float64
 12  carheight         205 non-null    float64
 13  curbweight        205 non-null    int64  
 14  enginetype        205 non-null    object 
 15  cylindernumber    205 non-null    object 
 16  enginesize        205 non-null    int64  
 1

In [48]:
df.describe()

,car_ID,symboling,wheelbase,carlength,carwidth,carheight,curbweight,enginesize,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
count,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000
mean,103.000000,0.834146,98.756585,174.049268,65.907805,53.724878,2555.565854,126.907317,3.329756,3.255415,10.142537,104.117073,5125.121951,25.219512,30.751220,13276.710571
std,59.322565,1.245307,6.021776,12.337289,2.145204,2.443522,520.680204,41.642693,0.270844,0.313597,3.972040,39.544167,476.985643,6.542142,6.886443,7988.852332
min,1.000000,-2.000000,86.600000,141.100000,60.300000,47.800000,1488.000000,61.000000,2.540000,2.070000,7.000000,48.000000,4150.000000,13.000000,16.000000,5118.000000
25%,52.000000,0.000000,94.500000,166.300000,64.100000,52.000000,2145.000000,97.000000,3.150000,3.110000,8.600000,70.000000,4800.000000,19.000000,25.000000,7788.000000
50%,103.000000,1.000000,97.000000,173.200000,65.500000,54.100000,2414.000000,120.000000,3.310000,3.290000,9.000000,95.000000,5200.000000,24.000000,30.000000,10295.000000
75%,154.000000,2.000000,102.400000,183.100000,66.900000,55.500000,2935.000000,141.000000,3.580000,3.410000,9.400000,116.000000,5500.000000,30.000000,34.000000,16503.000000
max,205.000000,3.000000,120.900000,208.100000,72.300000,59.800000,4066.000000,326.000000,3.940000,4.170000,23.000000,288.000000,6600.000000,49.000000,54.000000,45400.000000


In [49]:
df.isnull().sum()

car_ID              0
symboling           0
CarName             0
fueltype            0
aspiration          0
doornumber          0
carbody             0
drivewheel          0
enginelocation      0
wheelbase           0
carlength           0
carwidth            0
carheight           0
curbweight          0
enginetype          0
cylindernumber      0
enginesize          0
fuelsystem          0
boreratio           0
stroke              0
compressionratio    0
horsepower          0
peakrpm             0
citympg             0
highwaympg          0
price               0
dtype: int64

In [50]:
df.duplicated().sum()

np.int64(0)

* No duplicates data found

Categorical and numeric column

In [51]:
categorical=[x for x in x.columns if df[x].dtype=="object"]
print(categorical)

['CarName', 'fueltype', 'aspiration', 'doornumber', 'carbody', 'drivewheel', 'enginelocation', 'enginetype', 'cylindernumber', 'fuelsystem']


In [52]:
numerical=[x for x in x.columns if df[x].dtype=="int64"]
print(numerical)

['car_ID', 'symboling', 'curbweight', 'enginesize', 'horsepower', 'peakrpm', 'citympg', 'highwaympg']


In [53]:
print("Car Names :",df["CarName"].unique())
print("No. of CarNames : ",df["CarName"].value_counts().sum())


Car Names : ['alfa-romero giulia' 'alfa-romero stelvio' 'alfa-romero Quadrifoglio'
 'audi 100 ls' 'audi 100ls' 'audi fox' 'audi 5000' 'audi 4000'
 'audi 5000s (diesel)' 'bmw 320i' 'bmw x1' 'bmw x3' 'bmw z4' 'bmw x4'
 'bmw x5' 'chevrolet impala' 'chevrolet monte carlo' 'chevrolet vega 2300'
 'dodge rampage' 'dodge challenger se' 'dodge d200' 'dodge monaco (sw)'
 'dodge colt hardtop' 'dodge colt (sw)' 'dodge coronet custom'
 'dodge dart custom' 'dodge coronet custom (sw)' 'honda civic'
 'honda civic cvcc' 'honda accord cvcc' 'honda accord lx'
 'honda civic 1500 gl' 'honda accord' 'honda civic 1300' 'honda prelude'
 'honda civic (auto)' 'isuzu MU-X' 'isuzu D-Max ' 'isuzu D-Max V-Cross'
 'jaguar xj' 'jaguar xf' 'jaguar xk' 'maxda rx3' 'maxda glc deluxe'
 'mazda rx2 coupe' 'mazda rx-4' 'mazda glc deluxe' 'mazda 626' 'mazda glc'
 'mazda rx-7 gs' 'mazda glc 4' 'mazda glc custom l' 'mazda glc custom'
 'buick electra 225 custom' 'buick century luxus (sw)' 'buick century'
 'buick skyhawk' 'buick

In [54]:
print("Fuel Type",df["fueltype"].unique())
print("Aspiration",df["aspiration"].unique())
print("Door No.",df["doornumber"].unique())
print("Body",df["carbody"].unique())
print("DriverWheel",df["drivewheel"].unique())
print("enginetype",df["enginetype"].unique())
print("enginelocation",df["enginelocation"].unique())
print("cylindernumber",df["cylindernumber"].unique())
print("fuelsystem",df["fuelsystem"].unique())


Fuel Type ['gas' 'diesel']
Aspiration ['std' 'turbo']
Door No. ['two' 'four']
Body ['convertible' 'hatchback' 'sedan' 'wagon' 'hardtop']
DriverWheel ['rwd' 'fwd' '4wd']
enginetype ['dohc' 'ohcv' 'ohc' 'l' 'rotor' 'ohcf' 'dohcv']
enginelocation ['front' 'rear']
cylindernumber ['four' 'six' 'five' 'three' 'twelve' 'two' 'eight']
fuelsystem ['mpfi' '2bbl' 'mfi' '1bbl' 'spfi' '4bbl' 'idi' 'spdi']


**To find the relationship between categorical columns and price column(Different Groups) :Annova Test**

In [55]:
#Annova Test

related=[]
notRelated=[]

categorical_cols = df.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    groups = [group["price"].values
              for _, group in df.groupby(col)]

    if len(groups) > 1:
        f_stat, p_value = f_oneway(*groups)
        if p_value<0.05:
            related.append(col)
        else:
            notRelated.append(col)
    else:
        notRelated.append(col)
print("Related feature",related)
print("Non Related columns",notRelated)

Related feature ['CarName', 'aspiration', 'carbody', 'drivewheel', 'enginelocation', 'enginetype', 'cylindernumber', 'fuelsystem']
Non Related columns ['fueltype', 'doornumber']


* Drop the non related.

In [56]:
x=x.drop(columns=["fueltype","doornumber"],axis=1)
x

,car_ID,symboling,CarName,aspiration,carbody,drivewheel,enginelocation,wheelbase,carlength,carwidth,...,cylindernumber,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg
0,1,3,alfa-romero giulia,std,convertible,rwd,front,88.6,168.8,64.1,...,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27
1,2,3,alfa-romero stelvio,std,convertible,rwd,front,88.6,168.8,64.1,...,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27
2,3,1,alfa-romero Quadrifoglio,std,hatchback,rwd,front,94.5,171.2,65.5,...,six,152,mpfi,2.68,3.47,9.0,154,5000,19,26
3,4,2,audi 100 ls,std,sedan,fwd,front,99.8,176.6,66.2,...,four,109,mpfi,3.19,3.40,10.0,102,5500,24,30
4,5,2,audi 100ls,std,sedan,4wd,front,99.4,176.6,66.4,...,five,136,mpfi,3.19,3.40,8.0,115,5500,18,22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,201,-1,volvo 145e (sw),std,sedan,rwd,front,109.1,188.8,68.9,...,four,141,mpfi,3.78,3.15,9.5,114,5400,23,28
201,202,-1,volvo 144ea,turbo,sedan,rwd,front,109.1,188.8,68.8,...,four,141,mpfi,3.78,3.15,8.7,160,5300,19,25
202,203,-1,volvo 244dl,std,sedan,rwd,front,109.1,188.8,68.9,...,six,173,mpfi,3.58,2.87,8.8,134,5500,18,23
203,204,-1,volvo 246,turbo,sedan,rwd,front,109.1,188.8,68.9,...,six,145,idi,3.01,3.40,23.0,106,4800,26,27


**To find the relationship between numerical columns and price column(Continuous Value) : Pearson Coorelation**

In [57]:
#pearson Coorelation

related = []
not_related = []

for col in numerical:

    if col == "price":
        continue

    corr, p_value = pearsonr(df[col], df["price"])

    if p_value < 0.05:
        related.append(col)
    else:
        not_related.append(col)

print("Related:", related)
print("Not Related:", not_related)

Related: ['curbweight', 'enginesize', 'horsepower', 'citympg', 'highwaympg']
Not Related: ['car_ID', 'symboling', 'peakrpm']


* Drop non related array

In [58]:
x=x.drop(columns=['car_ID', 'symboling', 'peakrpm'],axis=1)
x

,CarName,aspiration,carbody,drivewheel,enginelocation,wheelbase,carlength,carwidth,carheight,curbweight,enginetype,cylindernumber,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,citympg,highwaympg
0,alfa-romero giulia,std,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,21,27
1,alfa-romero stelvio,std,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,21,27
2,alfa-romero Quadrifoglio,std,hatchback,rwd,front,94.5,171.2,65.5,52.4,2823,ohcv,six,152,mpfi,2.68,3.47,9.0,154,19,26
3,audi 100 ls,std,sedan,fwd,front,99.8,176.6,66.2,54.3,2337,ohc,four,109,mpfi,3.19,3.40,10.0,102,24,30
4,audi 100ls,std,sedan,4wd,front,99.4,176.6,66.4,54.3,2824,ohc,five,136,mpfi,3.19,3.40,8.0,115,18,22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,volvo 145e (sw),std,sedan,rwd,front,109.1,188.8,68.9,55.5,2952,ohc,four,141,mpfi,3.78,3.15,9.5,114,23,28
201,volvo 144ea,turbo,sedan,rwd,front,109.1,188.8,68.8,55.5,3049,ohc,four,141,mpfi,3.78,3.15,8.7,160,19,25
202,volvo 244dl,std,sedan,rwd,front,109.1,188.8,68.9,55.5,3012,ohcv,six,173,mpfi,3.58,2.87,8.8,134,18,23
203,volvo 246,turbo,sedan,rwd,front,109.1,188.8,68.9,55.5,3217,ohc,six,145,idi,3.01,3.40,23.0,106,26,27


**Encoding**
* **Car Names have different names more than 200.**
* **For encoding categorical value more than 200  different names Target Encoding is widely Used**

In [59]:
T_encoder=TargetEncoder(smooth="auto",cv=5,random_state=42,target_type="continuous")
y=pd.Series(y)
x["CarName"]=T_encoder.fit_transform(x[["CarName"]],y).ravel()
x


,CarName,aspiration,carbody,drivewheel,enginelocation,wheelbase,carlength,carwidth,carheight,curbweight,enginetype,cylindernumber,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,citympg,highwaympg
0,13312.089433,std,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,21,27
1,12975.610774,std,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,21,27
2,13312.089433,std,hatchback,rwd,front,94.5,171.2,65.5,52.4,2823,ohcv,six,152,mpfi,2.68,3.47,9.0,154,19,26
3,13256.665652,std,sedan,fwd,front,99.8,176.6,66.2,54.3,2337,ohc,four,109,mpfi,3.19,3.40,10.0,102,24,30
4,13312.089433,std,sedan,4wd,front,99.4,176.6,66.4,54.3,2824,ohc,five,136,mpfi,3.19,3.40,8.0,115,18,22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,12940.000000,std,sedan,rwd,front,109.1,188.8,68.9,55.5,2952,ohc,four,141,mpfi,3.78,3.15,9.5,114,23,28
201,13415.000000,turbo,sedan,rwd,front,109.1,188.8,68.8,55.5,3049,ohc,four,141,mpfi,3.78,3.15,8.7,160,19,25
202,15985.000000,std,sedan,rwd,front,109.1,188.8,68.9,55.5,3012,ohcv,six,173,mpfi,3.58,2.87,8.8,134,18,23
203,13312.089433,turbo,sedan,rwd,front,109.1,188.8,68.9,55.5,3217,ohc,six,145,idi,3.01,3.40,23.0,106,26,27


**Label Encoding**
* Categorical value other than CarName have only less different value (<20)
* Therefore Label encoding is widely used

In [60]:

L_encoder = LabelEncoder()
newCategorical=[y for y in x.columns if x[y].dtype=="object"]
for col in newCategorical:
    if col != "CarName":
        x[col] = L_encoder.fit_transform(x[col])

x.head()

,CarName,aspiration,carbody,drivewheel,enginelocation,wheelbase,carlength,carwidth,carheight,curbweight,enginetype,cylindernumber,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,citympg,highwaympg
0,13312.089433,0,0,2,0,88.6,168.8,64.1,48.8,2548,0,2,130,5,3.47,2.68,9.0,111,21,27
1,12975.610774,0,0,2,0,88.6,168.8,64.1,48.8,2548,0,2,130,5,3.47,2.68,9.0,111,21,27
2,13312.089433,0,2,2,0,94.5,171.2,65.5,52.4,2823,5,3,152,5,2.68,3.47,9.0,154,19,26
3,13256.665652,0,3,1,0,99.8,176.6,66.2,54.3,2337,3,2,109,5,3.19,3.40,10.0,102,24,30
4,13312.089433,0,3,0,0,99.4,176.6,66.4,54.3,2824,3,1,136,5,3.19,3.40,8.0,115,18,22


**Splitting of data**
* Splitting of data in a ratio of 8:2

In [61]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)


**<h1>Model Selection</h1>**



In [62]:
models={
    "LinearRegression":LinearRegression(),
    "Decision Tree":DecisionTreeRegressor(random_state=42),
    "Random Forest":RandomForestRegressor(
        n_estimators=100,
        random_state=42),
    "Gradient Boosting":GradientBoostingRegressor(n_estimators=100,random_state=42),
    "SVR":SVR()
}



**<h3>Criteria for best model :</h3>Less Mean absolute error,Mean squared error and highest R2Score.**  

In [63]:
result=[]
for name,model in models.items():
    model.fit(x_train,y_train)
    y_pred=model.predict(x_test)
    mse=mean_squared_error(y_test,y_pred)
    r2Score=r2_score(y_test,y_pred)
    mae=mean_absolute_error(y_test,y_pred)
    result.append([name,mae,mse,r2Score])

resultdf=pd.DataFrame(result,columns=["Model","MAE","MSE","R2Score"])
resultdf

,Model,MAE,MSE,R2Score
0,LinearRegression,2437.268267,1.506946e+07,0.809112
1,Decision Tree,1754.760171,6.454299e+06,0.918242
2,Random Forest,1279.947294,3.355130e+06,0.957500
3,Gradient Boosting,1660.487358,5.878093e+06,0.925541
4,SVR,5707.554989,8.701798e+07,-0.102275


In [64]:
print(resultdf.sort_values(by="MAE", ascending=True))

               Model          MAE           MSE   R2Score
2      Random Forest  1279.947294  3.355130e+06  0.957500
3  Gradient Boosting  1660.487358  5.878093e+06  0.925541
1      Decision Tree  1754.760171  6.454299e+06  0.918242
0   LinearRegression  2437.268267  1.506946e+07  0.809112
4                SVR  5707.554989  8.701798e+07 -0.102275


In [65]:
print(resultdf.sort_values(by="MSE", ascending=True))

               Model          MAE           MSE   R2Score
2      Random Forest  1279.947294  3.355130e+06  0.957500
3  Gradient Boosting  1660.487358  5.878093e+06  0.925541
1      Decision Tree  1754.760171  6.454299e+06  0.918242
0   LinearRegression  2437.268267  1.506946e+07  0.809112
4                SVR  5707.554989  8.701798e+07 -0.102275


In [66]:
print(resultdf.sort_values(by="R2Score", ascending=False))

               Model          MAE           MSE   R2Score
2      Random Forest  1279.947294  3.355130e+06  0.957500
3  Gradient Boosting  1660.487358  5.878093e+06  0.925541
1      Decision Tree  1754.760171  6.454299e+06  0.918242
0   LinearRegression  2437.268267  1.506946e+07  0.809112
4                SVR  5707.554989  8.701798e+07 -0.102275


**<h1>Model :** </h1>**<h2>Random Forest Regressor</h2>**

In [67]:
model=RandomForestRegressor(n_estimators=100,random_state=42)
model.fit(x_train,y_train)
y_pred=model.predict(x_test)
mse=mean_squared_error(y_test,y_pred)
r2Score=r2_score(y_test,y_pred)
mae=mean_absolute_error(y_test,y_pred)
print("MAE :",mae)
print("MSE :",mse)
print("R2 Score :",r2Score)

MAE : 1279.9472944250872
MSE : 3355130.107180232
R2 Score : 0.9574998832109807


**<h1>Parametric Tuning</h1>**
1. **GridSearchCV**

2. **RandomizedSearchCV**

<h1>GridSearchCV</h1>

In [ ]:
#GridSearchCV
parm_grid={
    "n_estimators":[100,200,300],
    "max_depth":[None,10,20,30],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4]
}
grid=GridSearchCV(estimator=model,param_grid=parm_grid,cv=5,scoring='r2',n_jobs=-1)
grid.fit(x_train,y_train)
print(grid.best_params_)

{'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 300}


<h1>RandomizedSearchCV</h1>

In [73]:

param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 10, 20, 30, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=50,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

random_search.fit(x_train, y_train)

print("Best Parameters:")
print(random_search.best_params_)

print("Best CV Score:")
print(random_search.best_score_)

Best Parameters:
{'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}
Best CV Score:
0.900623411580265


**GridSearchCV Best Model**


In [74]:
gscvbestModel=random_search.best_estimator_
gscvbestModel.fit(x_train,y_train)
ypred=gscvbestModel.predict(x_test)
mse=mean_squared_error(y_test,ypred)
r2Score=r2_score(y_test,ypred)
mae=mean_absolute_error(y_test,ypred)
print("MAE :",mae)
print("MSE :",mse)
print("R2 Score :",r2Score)

MAE : 1398.8891178861786
MSE : 5416332.295659999
R2 Score : 0.9313902150497673


**RandomizedSearchCV Best Model**

In [75]:
rscvbestModel=random_search.best_estimator_
rscvbestModel.fit(x_train,y_train)
ypred=rscvbestModel.predict(x_test)
mse=mean_squared_error(y_test,ypred)
r2Score=r2_score(y_test,ypred)
mae=mean_absolute_error(y_test,ypred)
print("MAE :",mae)
print("MSE :",mse)
print("R2 Score :",r2Score)

MAE : 1398.8891178861786
MSE : 5416332.295659999
R2 Score : 0.9313902150497673
